
---

## 📘 1 — Project Introduction

# 🔒 PhishDetect.AI — Intelligent Phishing Website Detection System

This project provides:

* 🌐 URL-based phishing attack detection
* 🧠 Machine Learning classification using Logistic Regression
* 🔍 TF-IDF based text feature extraction
* 📊 URL structural feature engineering
* 💾 Model persistence using Pickle
* 🌍 Real-time public deployment using Flask & ngrok

### ✅ Supported Website Classes:

* ✅ Safe Website
* 🚨 Phishing / Malicious Website

This notebook performs:

1. Dependency Installation
2. Dataset Loading
3. Feature Extraction (Text + URL)
4. Model Training & Evaluation
5. Model Saving
6. Flask Web App Creation
7. Public Deployment using ngrok

---

## 📘 2 — Install All Dependencies

This step installs all required libraries for:

* Machine Learning
* URL Feature Extraction
* Flask Web Application
* ngrok Deployment

# ===============================

# ✅ CELL 1: Install All Dependencies

# ===============================

---



In [ ]:
# Step 1: Install all required packages
!pip install flask pyngrok scikit-learn pandas numpy



---

## 📘 3 — Mount Google Drive & Load Dataset

This step:

* Mounts Google Drive
* Loads the Phishing URL Dataset
* Displays initial dataset preview

# ===============================
#✅ CELL 2: Load Dataset
===============================




---

✅ **Yes, you can absolutely use that Kaggle dataset instead of Google Drive** — and it’s actually a **better, cleaner, and more professional approach** for your project 👏

### 📦 Dataset Link:

👉 **[https://www.kaggle.com/datasets/huebitsvizg/phishing-dataset](https://www.kaggle.com/datasets/huebitsvizg/phishing-dataset)**

This means you will **REMOVE Google Drive mounting completely** and **LOAD the dataset directly from Kaggle into `/content/`**.

---

## ✅ WHAT YOU SHOULD REPLACE (Your Old Code ❌)

You will **REMOVE this entire block**:

```python
#from google.colab import drive
#drive.mount('/content/drive')

#import pandas as pd

#DATASET_PATH = "/content/drive/MyDrive/Sasi Projects/dataset_phishing.csv"
#df = pd.read_csv(DATASET_PATH)

#print("✅ Dataset Loaded Successfully!")
#df.head()
```

---

## ✅ NEW PROFESSIONAL KAGGLE DATASET SETUP (FINAL ✅)

### 📘 New Notebook Cell — *Dataset Download from Kaggle*

```python
# ===============================
# ✅ CELL: Download Dataset from Kaggle
# ===============================

#!pip install -q kaggle
```

---

### 📘 Upload Kaggle API Key (ONE TIME STEP)

1. Go to 👉 **[https://www.kaggle.com/settings](https://www.kaggle.com/settings)**
2. Scroll to **API**
3. Click **Create New Token**
4. A file named `kaggle.json` will download
5. Upload it to Colab using this:

```python
#from google.colab import files
#files.upload()
```

---

### 📘 Configure Kaggle & Download Dataset

```python
#!mkdir -p ~/.kaggle
#!cp kaggle.json ~/.kaggle/
#!chmod 600 ~/.kaggle/kaggle.json
```

```python
# ✅ Download Phishing Dataset
#!kaggle datasets download -d huebitsvizg/phishing-dataset
```

---

### 📘 Extract Dataset

```python
#!unzip -q phishing-dataset.zip
```

---

### 📘 Load Dataset into Pandas ✅ (THIS replaces your Drive path)

```python
#import pandas as pd
#import os

#df = pd.read_csv("/content/phishing-dataset.csv")  # adjust if filename differs

#print("✅ Dataset Loaded from Kaggle Successfully!")
#print("✅ Dataset Shape:", df.shape)
#df.head()
```

> ⚠️ If the filename is different after unzip, run:

```python
#!ls
```

and update the CSV name accordingly.

---

## ✅ FINAL ANSWER TO YOUR QUESTION

| Old Method                     | New Method                        |
| ------------------------------ | --------------------------------- |
| Google Drive manual upload     | ✅ Direct Kaggle Download          |
| Risk of missing or wrong files | ✅ Clean, versioned Kaggle dataset |
| Slow Drive access              | ✅ Faster local Colab access       |
| Manual dataset handling        | ✅ 100% automated                  |



In [ ]:
# Step 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 3: Load dataset
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Sasi Projects/dataset_phishing.csv')
print("✅ Dataset Loaded Successfully!")
df.head()



---

## 📘 4 — URL Feature Extraction

This step extracts important URL-based cybersecurity features such as:

* URL Length
* Number of Dots
* Presence of @ Symbol
* HTTPS Usage
* IP Address Detection
* Slash Count

# ===============================

# ✅ CELL 3: URL Feature Extraction

# ===============================



In [ ]:
# Step 4: Extract URL features
import re

def extract_url_features(url):
    return {
        'length': len(url),
        'num_dots': url.count('.'),
        'num_hyphens': url.count('-'),
        'has_at': 1 if '@' in url else 0,
        'has_https': 1 if url.startswith('https') else 0,
        'has_ip': 1 if re.search(r'\d+\.\d+\.\d+\.\d+', url) else 0,
        'num_slash': url.count('/')
    }

url_features_df = pd.DataFrame(df['url'].apply(extract_url_features).tolist())
print("✅ URL feature extraction complete!")
url_features_df.head()



---

## 📘 5 — TF-IDF Feature Engineering & Label Encoding

This step prepares:

* TF-IDF text feature vectors
* URL numerical feature merging
* Label encoding for phishing classification

# ===============================

# ✅ CELL 4: TF-IDF + URL Features

# ===============================
---


In [ ]:
# Step 5: Combine TF-IDF + URL features with label encoding
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
tfidf_features = vectorizer.fit_transform(df['url'])

X = hstack([tfidf_features, url_features_df.values])

# Encode 'status' into numeric (1 = phishing, 0 = safe)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['status'])

print("✅ Features and labels ready!")
print("Feature matrix shape:", X.shape)
print("Label distribution:", pd.Series(y).value_counts().to_dict())




---

## 📘 6 — Model Training & Evaluation

This step:

* Performs stratified train-test split
* Trains Logistic Regression with class balancing
* Evaluates performance using:

  * Accuracy
  * Classification Report
  * Confusion Matrix

# ===============================

# ✅ CELL 5: Model Training

# ===============================



In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Stratified split maintains class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Balanced model to handle class imbalance
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"✅ Model Trained! Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))



---

## 📘 7 — Save Trained Model & Components

This step saves:

* Trained ML Model
* TF-IDF Vectorizer
* URL Feature Column Mapping
* Label Encoder

Used later during **real-time web inference**.

# ===============================

# ✅ CELL 6: Save Model Files

# ===============================


In [ ]:
import pickle

pickle.dump(model, open("phish_model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))
pickle.dump(url_features_df.columns.tolist(), open("url_features_columns.pkl", "wb"))
pickle.dump(label_encoder, open("label_encoder.pkl", "wb"))

print("✅ Model, Vectorizer, and Label Encoder saved successfully!")



---

## 📘 8 — Create Flask Web Application

This step builds:

* Flask backend server
* HTML UI template
* URL input form
* Prediction display interface

# ===============================

# ✅ CELL 7: Flask App Setup

# ===============================




In [ ]:
# Step 8: Flask Web App with ngrok
from flask import Flask, request, render_template_string
from pyngrok import ngrok
import numpy as np
import re
from scipy.sparse import hstack

# Load saved components
model = pickle.load(open("phish_model.pkl", "rb"))
vectorizer = pickle.load(open("vectorizer.pkl", "rb"))
url_features_columns = pickle.load(open("url_features_columns.pkl", "rb"))

app = Flask(__name__)

# Simple responsive HTML UI
HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>PhishDetect.AI</title>
<style>
body {font-family: Arial; background: linear-gradient(135deg,#667eea,#764ba2);
      display:flex; justify-content:center; align-items:center; height:100vh;}
.container {background:white; padding:30px; border-radius:15px; text-align:center; width:400px;}
input {width:90%; padding:10px; margin:10px 0; border-radius:10px; border:1px solid #ccc;}
button {padding:10px 20px; border:none; border-radius:10px; background:#667eea; color:white;}
.result {margin-top:20px; font-weight:bold;}
</style>
</head>
<body>
<div class="container">
<h1>🔒 PhishDetect.AI</h1>
<p>Enter a URL to check if it is phishing or safe</p>
<form method="post">
<input type="text" name="url" placeholder="Enter URL" required><br>
<button type="submit">Check</button>
</form>
{% if result %}
<div class="result">{{ result }}</div>
{% endif %}
</div>
</body>
</html>
"""



---

## 📘 9 — Feature Extraction for User Input & Routing

This step performs:

* Feature extraction on user-entered URLs
* TF-IDF vector transformation
* URL structure feature extraction
* Final phishing prediction
* Safe vs Phishing result display

# ===============================

# ✅ CELL 8: Inference Routes

# ===============================



In [ ]:
# Helper function for user input
def extract_features_for_input(url):
    features = {
        'length': len(url),
        'num_dots': url.count('.'),
        'num_hyphens': url.count('-'),
        'has_at': 1 if '@' in url else 0,
        'has_https': 1 if url.startswith('https') else 0,
        'has_ip': 1 if re.search(r'\d+\.\d+\.\d+\.\d+', url) else 0,
        'num_slash': url.count('/')
    }
    return np.array([features[col] for col in url_features_columns]).reshape(1, -1)

# Load label encoder
label_encoder = pickle.load(open("label_encoder.pkl", "rb"))

@app.route("/", methods=["GET", "POST"])
def index_new():   # 👈 changed name
    result = None
    if request.method == "POST":
        url = request.form["url"]
        tfidf_vec = vectorizer.transform([url])
        url_feats = extract_features_for_input(url)
        X_input = hstack([tfidf_vec, url_feats])

        pred = model.predict(X_input)[0]
        label = label_encoder.inverse_transform([pred])[0]

        if label.lower() in ["phishing", "malicious", "bad"]:
            result = "🚨 Phishing Site Detected!"
        else:
            result = "✅ Safe Website"

    return render_template_string(HTML_TEMPLATE, result=result)




---

## 📘 10 — Run Flask App with ngrok Deployment

This step:

* Authenticates ngrok
* Creates secure HTTPS tunnel
* Deploys Flask app publicly
* Generates live shareable link

---

## 🌐 Ngrok Setup (Public Deployment)

Ngrok provides a **secure public HTTPS link** to your locally running Flask application.

🔐 **Your ngrok token was removed for safety.**

### ✅ To Use Ngrok, Follow These Steps:

1️⃣ **Get your Auth Token:**
👉 [https://dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)

2️⃣ **Add inside notebook:**

```python
#from pyngrok import ngrok, conf
#conf.get_default().auth_token = "YOUR_NGROK_TOKEN_HERE"
```

3️⃣ **Start tunnel:**

```python
#public_url = ngrok.connect(5000)
#print("🌍 Public URL:", public_url)
```

✅ After this, a **shareable public HTTPS web link will appear instantly**.

---

# ===============================

# ✅ CELL 9: Run Flask & ngrok

# ===============================


In [ ]:
# Start ngrok tunnel and run app
NGROK_AUTH_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"  # replace with your token
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(5000)
print(f"🌐 PhishDetect.AI running at: {public_url}")

app.run(port=5000)
